<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Lab2aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Report

* How to run the code
This code is written to run in Google Colab or in a regular Jupyter Notebook. To run the code, you just need to run the cells in order from top to bottom. The dataset is handled automatically by the code. It downloads the hymenoptera_data (images of ants and bees) via a script cell, unpacks it, and places it in a local folder. No manual configuration or file upload is required.

---

* Description of the dataset

Dataset: I have used the Hymenoptera dataset, which contains photographs of ants and bees. The images are in color (RGB) and have been transformed to a resolution of 224x224 pixels to fit the models.

Problem Type: This is a classification problem. The goal is for the network to categorize each image into one of two classes: ant or bee. It is not a regression problem because we are predicting a category (label) and not a continuous numerical value.

---
* Libraries and versions

The following libraries are used in the solution:

> torch (PyTorch) - The main framework for deep learning.

> torchvision - Used to access pre-trained models (ResNet 18, MobileNetV2) as well as for image transformations and data augmentation.

> matplotlib - For visualization.

> os, time, copy - For file management and time meausurement during training.

These libraries are pre-installed in Google Colab. It running locally, they can be installed via pip install torch torchvision.

---

* Custom folder structurees

Data storage: The code uses the ./hymenoptera_data folder that is automatically createdin the working directory by the initial commands (!wget and !unzip).

External files: No external files need to be downloaded or uploaded manually, as the script retrieves the data directly from PyTorch's official server.


---

* Complementary information (Architecture and results)

In this lab I have completed the following subtasks:

> Data augmentation:
To prevent overfitting, I have implemented techniques such as RandomResizedCrop, RandomHorizontalFlip and RandomRotation (15 degrees) for the training data.

> CNN vs MLP:
I have designed and compared two models. The MLP model (with a hidden layer of 512 neurons) performed worse as it cannot handle the spatial sctructure of images. The SImpleCNN (two convolutional layers with Max-Pooling) showed much better stability and learning progress.

> Fine-tuning of pre-trained models:

>> ResNet 18: Implemented by "freezing" th eweights of the convolutional base and only training the final fully connected layer.

>> MobileNetV2: Implemented by reconstructing the classifier layer and fine-tuning the entire network's weights.

Results: The comparison between the architectures reveals that the MLP Model performed worst, stagnating at a validation accuracy of




## Setup and Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy

# use GPU (CUDA) if available, otherwise CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda:0


## Data Augmentation and Dataloaders

In [ ]:
# define transformations for training (with augmentation) and validation (standardized)

data_transforms = {
    'train': transforms.Compose([          # transformations for the training set
        transforms.RandomResizedCrop(224), # crop image to 224x224 at random scales
        transforms.RandomHorizontalFlip(), # flip image horizontally with 50% probability
        transforms.RandomRotation(15),     # rotate image by up to 15 degrees
        transforms.ToTensor(),             # convert image to PyTorch Tensor
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # standard ImageNet normalization
    ]),
    'val': transforms.Compose([       # transformations for the validation set
        transforms.Resize(256),       # resize smaller edge to 256
        transforms.CenterCrop(224),   # crop the center 224x224 area
        transforms.ToTensor(),        # convert to tensor
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # normalize
    ]),
}





## Download dataset

In [ ]:
if not os.path.exists('hymenoptera_data'):                           # check if dataset folder exists
    !wget https://download.pytorch.org/tutorial/hymenoptera_data.zip # download zip file
    !unzip hymenoptera_data.zip                                      # extract the zip file
    !rm hymenoptera_data.zip                                         # delete the zip file ater extraction

data_dir = 'hymenoptera_data'  # root directory for the data

# load images from folders using ImageFolder
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}

# create data loaders to provide batches of data
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=2) for x in ['train', 'val']}

# store the total number of image in each split
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

# extract class names ('ants' and 'bees')
class_names = image_datasets['train'].classes

## Training loop

In [ ]:
def train_model(model, criterion, optimizer, scheduler=None, num_epochs=3):
  for epoch in range(num_epochs):   # loop ver the dataset multiple times
    print(f'Epoch {epoch}/{num_epochs - 1}') # print current epoch progress
    print('-' * 10)

    for phase in ['train', 'val']: # each epoch has a training and validation phase
      if phase == 'train':
        model.train() # set to training mode
      else:
        model.eval() # set to evaluation mode

      running_loss = 0.0 # accumulator for loss
      running_corrects = 0 # accumulator for correct predictions

      for inputs, labels in dataloaders[phase]: # iterate over batches of data
        inputs = inputs.to(device) # move images to device
        labels = labels.to(device) # move labels
        optimizer.zero_grad() # clear previous gradients

        with torch.set_grad_enabled(phase == 'train'): # only track gradients in training
          outputs = model(inputs) # forward pass: compute model output
          _, preds = torch.max(outputs, 1) # get the index of the highest probability
          loss = criterion(outputs, labels) # calculate loss based on target labels

          if phase == 'train':
            loss.backward() # compute graidents (backpropagation)
            optimizer.step() # update model parameters

        running_loss += loss.item() * inputs.size(0) # update total loss
        running_corrects += torch.sum(preds == labels.data) # update total correct count

      if phase == 'train' and scheduler is not None: # if a scheduler exists
        scheduler.step() # update the learning rate

      epoch_loss = running_loss / dataset_sizes[phase] # calculate average loss for the epoch
      epoch_acc = running_corrects.double() / dataset_sizes[phase] # calculate accuracy

      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

  return model # returned the trained model

## Design: CNN vs MLP

MLP Model

In [ ]:
class MLPModel(nn.Module):
  def __init__(self):
    super(MLPModel, self).__init__()    # initialize the parent class
    self.flatten = nn.Flatten()         # layer to convert 3D image to 1D vector
    self.fc = nn.Sequential(            # group layers in sequence
        nn.Linear(224*224*3, 512),      # input layer: flattened pizels to 512 neurons
        nn.ReLU(),                      # activation function for non-linearity
        nn.Linear(512, len(class_names)) # output layer: 512 to number of classes
    )

  def forward(self, x):                 # define the forward pass
    x = self.flatten(x)                 # flatten the input image
    return self.fc(x)                   # pass through fully connected layers

Custom CNN Model

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()   # initialize the parent class
    self.features = nn.Sequential (     # convolutional layers for feature extraction
        nn.Conv2d(3, 16, kernel_size=3, padding=1),  # conv layer: 3 channels to 16 filters
        nn.ReLU(inplace=True),  #ReLu activation
        nn.MaxPool2d(2, 2),  # downsample image by factor of 2
        nn.Conv2d(16, 32, kernel_size=3, padding=1), # conv layer: 16 to 32 filters
        nn.ReLU(), # ReLu activation
        nn.MaxPool2d(2, 2) # downsample again
    )
    self.classifier = nn.Sequential( # fully connected layers for classification
        nn.Linear(32*56*56, 128), # input from conv layers to 128 neurons
        nn.ReLU(), # ReLu activation
        nn.Linear(128, len(class_names)) # final output layer
    )

  def forward(self, x): # define the forward pass
    x = self.features(x) # extract spatial features
    x = x.view(x.size(0), -1) # reshape (flatten) for the classifier
    return self.classifier(x) # classify based on extracted features


In [ ]:
criterion = nn.CrossEntropyLoss() # standard loss function for classification

print("--- Training MLP Model ---")
model_mlp = MLPModel().to(device) # instantiate MLP and move to device
optimizer_mlp = optim.SGD(model_mlp.parameters(), lr=0.001, momentum=0.9) # define optimizer
train_model(model_mlp, criterion, optimizer_mlp, num_epochs=3); # execute training

print("\n--- Training Simple CNN Model ---")
model_cnn = SimpleCNN().to(device) # instantiate CNN and move to device
optimizer_cnn = optim.SGD(model_cnn.parameters(), lr=0.001, momentum=0.9) # define optimizer
train_model(model_cnn, criterion, optimizer_cnn, num_epochs=3); # execute training




--- Training MLP Model ---
Epoch 0/2
----------
train Loss: 9.4291 Acc: 0.5492
val Loss: 47.9533 Acc: 0.5359
Epoch 1/2
----------
train Loss: 34.4876 Acc: 0.5164
val Loss: 65.6406 Acc: 0.5948
Epoch 2/2
----------
train Loss: 136.3269 Acc: 0.4754
val Loss: 738.6536 Acc: 0.5229

--- Training Simple CNN Model ---
Epoch 0/2
----------
train Loss: 0.6921 Acc: 0.5451
val Loss: 0.7936 Acc: 0.4575
Epoch 1/2
----------
train Loss: 0.6793 Acc: 0.5697
val Loss: 0.6933 Acc: 0.5817
Epoch 2/2
----------
train Loss: 0.6524 Acc: 0.6025
val Loss: 0.6524 Acc: 0.5621


## Fine-tuning Pretrained Models

Method 1: Freezing Weights (ResNet18)

In [ ]:
print("\n--- Training ResNet18 ---")
model_resnet = models.resnet18(pretrained=True) # load a model with pre-trained ImageNet weights
for param in model_resnet.parameters(): # loop through all model parameters
  param.requires_grad = False # freeze parameters so they don't update

num_ftrs = model_resnet.fc.in_features # get input size of the original final layer
model_resnet.fc = nn.Linear(num_ftrs, len(class_names)) # replace it with a new layer for task
model_resnet = model_resnet.to(device) # move model to device

# only optimize the parameters of the new final layer
optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
train_model(model_resnet, criterion, optimizer_res, num_epochs=3);


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



--- Training ResNet18 ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Epoch 0/2
----------
train Loss: 0.6440 Acc: 0.6475
val Loss: 0.2954 Acc: 0.9020
Epoch 1/2
----------
train Loss: 0.4521 Acc: 0.7787
val Loss: 0.2753 Acc: 0.8889
Epoch 2/2
----------
train Loss: 0.5434 Acc: 0.7377
val Loss: 0.2685 Acc: 0.9281


Method 2: Reconstructing Layers (MobileNetV2)

In [ ]:
print("\n--- Training MobileNet V2 ")
model_mobilenet = models.mobilenet_v2(pretrained=True) # load pre-trained MobileNetV2
num_ftrs = model_mobilenet.classifier[1].in_features # get iinput size of the classifier
model_mobilenet.classifier[1] = nn.Linear(num_ftrs, len(class_names)) # replace output layer
model_mobilenet = model_mobilenet.to(device) # move model to device

# optimize all parameters (fine-tune the whole network)
optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
train_model(model_mobilenet, criterion, optimizer_mob,num_epochs=3);

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



--- Training MobileNet V2 
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 97.3MB/s]

Epoch 0/2
----------


train Loss: 0.5403 Acc: 0.7008
val Loss: 0.4004 Acc: 0.9020
Epoch 1/2
----------
train Loss: 0.8775 Acc: 0.7172
val Loss: 0.3857 Acc: 0.8105
Epoch 2/2
----------
train Loss: 0.6185 Acc: 0.7131
val Loss: 0.3867 Acc: 0.8301
